# Intelligence Lab — Random Forest experiment

Temporary JupyterLite-friendly notebook. This reproduces the same synthetic fraud dataset and split as the main experiment. It uses NumPy, pandas, Matplotlib and scikit-learn only; no imbalanced-learn dependency.

## Research question
Can a nonlinear tree-based model identify fraud more effectively than Logistic Regression?

## Evaluation discipline
- Fit the model on a fit split created from the original training data.
- Choose a threshold using validation data only.
- Evaluate the selected threshold once on the untouched test set.
- Do not tune again using test results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, average_precision_score
)

RANDOM_STATE = 42
print('Imports ready')

## 1. Recreate the dataset and splits
This uses the same data-generation settings as the main fraud notebook. The final test set stays untouched while we train and select a threshold.


In [ ]:
X, y = make_classification(
    n_samples=10_000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_clusters_per_class=2,
    weights=[0.99, 0.01],
    flip_y=0.001,
    class_sep=1.0,
    random_state=RANDOM_STATE
)

feature_names = [
    'transaction_amount', 'transactions_per_hour', 'account_age_days',
    'distance_from_usual_km', 'failed_attempts', 'merchant_risk',
    'device_change_score', 'hour_of_day'
]
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name='is_fraud')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y_train
)

print('Dataset:', X.shape)
print('Fit:', X_fit.shape, '| Validation:', X_val.shape, '| Test:', X_test.shape)
print('\nFraud counts — fit / validation / test:')
print(y_fit.sum(), y_val.sum(), y_test.sum())

## 2. Train Random Forest
Trees split on feature values, so feature standardisation is generally not required. `class_weight='balanced'` increases the penalty for mistakes on the minority class.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_fit, y_fit)
print('Random Forest trained.')

## 3. Select a threshold on validation data only
The threshold controls how much fraud probability is needed to raise an alert. We will inspect the trade-off and select a threshold without touching the test set.


In [ ]:
val_scores = rf_model.predict_proba(X_val)[:, 1]
threshold_rows = []

for threshold in np.arange(0.05, 1.00, 0.05):
    pred = (val_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'fraud_alerts': int(fp + tp)
    })

threshold_results = pd.DataFrame(threshold_rows)
print('Validation PR-AUC:', round(average_precision_score(y_val, val_scores), 4))
threshold_results

In [ ]:
ax = threshold_results.plot(x='threshold', y=['precision', 'recall'], marker='o')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Random Forest — validation precision and recall')
plt.grid(True)
plt.show()

threshold_results.plot(
    x='threshold', y='fraud_alerts', marker='o', legend=False,
    title='Random Forest — validation fraud alerts by threshold'
)
plt.ylabel('Number of alerts')
plt.grid(True)
plt.show()

## 4. Choose a threshold using the exercise constraints
Target: recall at least 90% and no more than 100 alerts. If no threshold meets both, report that honestly and select a best-effort threshold under the alert cap for diagnosis.


In [ ]:
MIN_RECALL = 0.90
MAX_ALERTS = 100

feasible = threshold_results[
    (threshold_results['recall'] >= MIN_RECALL) &
    (threshold_results['fraud_alerts'] <= MAX_ALERTS)
]

if not feasible.empty:
    selected_threshold = float(feasible.sort_values('threshold').iloc[-1]['threshold'])
    print('A validation threshold satisfies both targets.')
else:
    print('No tested validation threshold satisfies both targets.')
    under_cap = threshold_results[threshold_results['fraud_alerts'] <= MAX_ALERTS]
    if not under_cap.empty:
        best_row = under_cap.sort_values(['recall', 'threshold'], ascending=[False, False]).iloc[0]
    else:
        best_row = threshold_results.sort_values(['fraud_alerts', 'recall'], ascending=[True, False]).iloc[0]
    selected_threshold = float(best_row['threshold'])
    print('Using a best-effort threshold for diagnosis.')

print('Selected threshold:', selected_threshold)
threshold_results.loc[threshold_results['threshold'] == selected_threshold]

## 5. Final evaluation on untouched test data
Apply the selected threshold once. Do not use these test results to choose a new threshold.


In [ ]:
test_scores = rf_model.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= selected_threshold).astype(int)

print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test, test_pred))
print('\nClassification report:')
print(classification_report(y_test, test_pred, zero_division=0))

tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()
alerts = int(fp + tp)
print('Test PR-AUC:', round(average_precision_score(y_test, test_scores), 4))
print('Total alerts:', alerts)
print('Recall target met:', recall_score(y_test, test_pred, zero_division=0) >= MIN_RECALL)
print('Alert target met:', alerts <= MAX_ALERTS)

## 6. Quick interpretation

Record your answers:
1. Did Random Forest improve recall compared with class-weighted Logistic Regression (previous result: 90.48% recall, 510 alerts at the default threshold)?
2. How many false positives did it produce?
3. Did the validation threshold satisfy both constraints?
4. Did the test result support the validation-based choice?
5. Is Random Forest a better candidate based on PR-AUC and the confusion matrix?

**Caution:** This is synthetic data with very few fraud examples. Results are for learning the evaluation workflow, not real-world banking estimates.
